## 1. Configuración del entorno

In [3]:
# Importar el paquete GeoAI
import geoai

In [4]:
# Especificar las carpetas
results_path = "../geoai/results"
models_path = "../geoai/models"
output_path = f"{results_path}/output"

## 2. Segmentación de los casos de estudio

In [3]:
# Inferencia sobre el área de interés en 2021
geoai.semantic_segmentation(
    input_path=f"{results_path}/area_estudio_2021.tif",
    output_path=f"{output_path}/area_estudio_2021_prediction.tif",
    model_path=f"{models_path}/unet_efficientnet/best_model.pth",
    architecture="unet",
    encoder_name="efficientnet-b1",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=8,
    probability_threshold=0.3,
    quiet=True,
)

In [4]:
# Inferencia sobre el área de interés en 2024
geoai.semantic_segmentation(
    input_path=f"{results_path}/area_estudio_2024.tif",
    output_path=f"{output_path}/area_estudio_2024_prediction.tif",
    model_path=f"{models_path}/unet_efficientnet/best_model.pth",
    architecture="unet",
    encoder_name="efficientnet-b1",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=8,
    probability_threshold=0.3,
    quiet=True,
)

## 3.  Vectorización de las máscaras

In [7]:
# Vectorización inicial de la predicción de 2021
raster_path_2021 = f"{output_path}/area_estudio_2021_prediction.tif"
vector_path_2021 = f"{output_path}/area_estudio_2021_prediction.geojson"

gdf = geoai.orthogonalize(raster_path_2021, vector_path_2021, epsilon=2)
vector_2021_props = geoai.add_geometric_properties(gdf, area_unit="m2", length_unit="m")

Processing 9681 features...


Converting features: 100%|████████████| 9681/9681 [00:00<00:00, 29434.94shape/s]


Saving to ../geoai/results/output/area_estudio_2021_prediction.geojson...
Done!


In [8]:
# Vectorización inicial de la predicción de 2021
raster_path_2024 = f"{output_path}/area_estudio_2024_prediction.tif"
vector_path_2024 = f"{output_path}/area_estudio_2024_prediction.geojson"

gdf = geoai.orthogonalize(raster_path_2024, vector_path_2024, epsilon=2)
vector_2024_props = geoai.add_geometric_properties(gdf, area_unit="m2", length_unit="m")

Processing 3261 features...


Converting features: 100%|████████████| 3261/3261 [00:00<00:00, 30016.91shape/s]


Saving to ../geoai/results/output/area_estudio_2024_prediction.geojson...
Done!


In [18]:
# Visualizar las primeras predicciones
import ipywidgets as w
import leafmap.leafmap as leafmap

data = [
    (
        vector_2021_props,
        "orange",
        "Predicción 2021",
        "https://idecan1.grafcan.es/ServicioWMS/Historico/Ortofotos/OrtoExpress_2021?",
        "IDECanarias OrtoExpress 2021 (20 cm/pixel)",
    ),
    (
        vector_2024_props,
        "red",
        "Predicción 2024",
        "https://idecan1.grafcan.es/ServicioWMS/OrtoExpress?",
        "IDECanarias Ortofoto Territorial Campaña 2024",
    ),
]

maps = [
    leafmap.Map(
        center=[28.6162, -17.8986],
        zoom=18,
        min_zoom=17,
        max_zoom=20,
        toolbar_control=False,
        draw_control=False,
        fullscreen_control=False,
        layers_control=True,
    )
    for i in range(2)
]

for m, (geojson, color, model_name, wms_url, wms_layer_name) in zip(maps, data):
    m.clear_layers()

    m.add_wms_layer(
        url=wms_url,
        layers="WMS_OrtoExpress",
        name=wms_layer_name,
        attribution='<a href="https://www.grafcan.es/aviso-legal/" target="_blank">GRAFCAN</a>, Ortofotos Express de Canarias',
        format="image/jpeg",
        max_zoom=20,
        base=True,
    )

    m.add_gdf(
        geojson,
        layer_name=model_name,
        style={"color": color, "fillOpacity": 0.25, "weight": 2},
        zoom_to_layer=True,
    )

w.jslink((maps[0], "center"), (maps[1], "center"))
w.jslink((maps[0], "zoom"), (maps[1], "zoom"))

maps[0].layout.width = maps[1].layout.width = "50%"
display(w.HBox(maps, layout=w.Layout(height="600px")))

## 4. Filtrado y regularización

In [41]:
# Filtro de figuras de menos de 15 metros cuadrados
vector_2021_filter = vector_2021_props[(vector_2021_props["area_m2"] > 15)]
vector_2024_filter = vector_2024_props[(vector_2024_props["area_m2"] > 15)]

# Regularización de las formas de las huellas
area_estudio_2021 = geoai.adaptive_regularization(
    vector_2021_filter, simplify_tolerance=0.5, area_threshold=0.85, preserve_shape=True
)
area_estudio_2024 = geoai.adaptive_regularization(
    vector_2024_filter, simplify_tolerance=0.5, area_threshold=0.85, preserve_shape=True
)

In [49]:
# Guardado de las huellas
geoai.add_geometric_properties(
    area_estudio_2021, area_unit="m2", length_unit="m"
).to_file(f"{output_path}/area_estudio_2021.geojson", driver="GeoJSON")
geoai.add_geometric_properties(
    area_estudio_2024, area_unit="m2", length_unit="m"
).to_file(f"{output_path}/area_estudio_2024.geojson", driver="GeoJSON")

## 5. Comprobación de la huella final

In [50]:
# Visualización final de las huellas extraidas previo y posterior al desastre
import ipywidgets as w
import leafmap.leafmap as leafmap

data = [
    (
        area_estudio_2021,
        "orange",
        "Predicción reglarizada 2021",
        "https://idecan1.grafcan.es/ServicioWMS/Historico/Ortofotos/OrtoExpress_2021?",
        "IDECanarias OrtoExpress 2021 (20 cm/pixel)",
    ),
    (
        area_estudio_2024,
        "red",
        "Predicción regularizada 2024",
        "https://idecan1.grafcan.es/ServicioWMS/OrtoExpress?",
        "IDECanarias Ortofoto Territorial Campaña 2024",
    ),
]

maps = [
    leafmap.Map(
        center=[28.6162, -17.8986],
        zoom=18,
        min_zoom=16,
        max_zoom=22,
        toolbar_control=False,
        draw_control=False,
        fullscreen_control=False,
        layers_control=True,
    )
    for i in range(2)
]

for m, (geojson, color, model_name, wms_url, wms_layer_name) in zip(maps, data):
    m.clear_layers()

    m.add_wms_layer(
        url=wms_url,
        layers="WMS_OrtoExpress",
        name=wms_layer_name,
        attribution='<a href="https://www.grafcan.es/aviso-legal/" target="_blank">GRAFCAN</a>, Ortofotos Express de Canarias',
        format="image/jpeg",
        max_zoom=22,
        base=True,
    )

    m.add_gdf(
        geojson,
        layer_name=model_name,
        style={"color": color, "fillOpacity": 0.25, "weight": 2},
        zoom_to_layer=False,
    )

w.jslink((maps[0], "center"), (maps[1], "center"))
w.jslink((maps[0], "zoom"), (maps[1], "zoom"))

maps[0].layout.width = maps[1].layout.width = "50%"
display(w.HBox(maps, layout=w.Layout(height="600px")))

In [59]:
# Calcular conteo de huellas
num_2021 = len(area_estudio_2021)
num_2024 = len(area_estudio_2024)
diferencia = num_2021 - num_2024

# Imprimir el resultado
print(f"Edificios en 2021: {num_2021}")
print(f"Edificios en 2024: {num_2024}")
print(f"{'='*60}")
print(f"Variación:  {diferencia} edificios perdidos")

Edificios en 2021: 4345
Edificios en 2024: 1600
Variación:  2745 edificios perdidos
